In [0]:
from pyspark.sql.functions import col, hour, count, avg, max as spark_max

silver_df = spark.table("main.aviation_departures.silver_cle_departures")
display(silver_df)

In [0]:
silver_enriched = silver_df.withColumn(
    "dep_hour_utc",
    hour(col("dep_scheduled_utc"))
)

display(silver_enriched)

In [0]:
gold_df = (
    silver_enriched.groupBy(
        "flight_date",
        "dep_hour_utc",
        "airline_name",
        "dep_iata",
        "arr_iata"
    )
    .agg(
        count("*").alias("scheduled_departure_count"),
        avg("dep_delay").alias("avg_dep_delay_minutes"),
        spark_max("dep_delay").alias("max_dep_delay_minutes")
    )
)

display(gold_df)

In [0]:
gold_df.write.format("delta").mode("overwrite").saveAsTable("main.aviation_departures.gold_cle_departure_metrics")